# notebook_ref_source_doctype_map

Creates and seeds `genealogy.ref_source_doctype_map`, the hand-maintained lookup from an Ancestry source title to a `doc_type_category`. `genealogy.v_source_doctype_map` is a thin view over it (defined in `views_document_and_fact_review`) and `refresh_gold_source_coverage` joins that view.

**The table is never rebuilt.** Cell 1 is `CREATE TABLE IF NOT EXISTS`; never `CREATE OR REPLACE` or `TRUNCATE` it. After this one-off setup, add or change titles with `INSERT` / `UPDATE` through the MCP `execute_write_sql` (fully qualified name, single line), not by editing a notebook. Find candidates in `gold_housekeeping_unmapped_source_titles`.

Run order: this notebook (once), then `views_document_and_fact_review` Cell 7 (repoint the view), then `refresh_gold_source_coverage`, notebook 03, notebook B. All cells are idempotent.

Rules:
- `source_title` is the key and must match the citation title exactly (case, punctuation, en dashes).
- Categories that matter to `refresh_gold_source_coverage`: `BirthIndex` / `MarriageIndex` / `DeathIndex` give INDEX_ONLY; `OTHER_NO_FILE` gives NOT_APPLICABLE; any other category with no matched document is UNCOVERED. A category must equal the `doc_type_detected` of the file to match it, and that comparison is case-sensitive (`Other` is not `OTHER`).
- Do not map a combined-collection title (baptisms, marriages and burials under one title) to a single category: it would miscategorise the other event types. Leave those unmapped.

## History
- 2026-10-16: created, replacing the hardcoded VALUES list in `v_source_doctype_map`. Cell 2 seeds the 117 rows the view held. Cell 3 applies the Asana 1218767063521380 changes: 23 new single-purpose titles (incl. the two Scotland statutory titles and Lunacy as `Other`) and `Web: UK, Burial and Cremation Index` moved from `BurialRegister` to `DeathIndex` (an index; add a separate citation if register images are ever bought).


In [0]:
%sql
-- Never replace or truncate: rows are maintained by hand (see the header).
CREATE TABLE IF NOT EXISTS genealogy.ref_source_doctype_map (
  source_title      STRING    NOT NULL COMMENT 'Ancestry source title, exact match; the key',
  doc_type_category STRING    NOT NULL COMMENT 'Category used by refresh_gold_source_coverage (doc_type_detected vocabulary, plus BirthIndex/MarriageIndex/DeathIndex/OTHER_NO_FILE)',
  notes             STRING             COMMENT 'Why mapped this way / which task',
  added_at          TIMESTAMP          COMMENT 'When the row was added'
) COMMENT 'Hand-maintained lookup: source title to doc type category. Never rebuilt; edit via INSERT/UPDATE.';

In [0]:
%sql
-- One-off seed: the rows v_source_doctype_map held on 2026-10-16. Idempotent (skips titles already present).
INSERT INTO genealogy.ref_source_doctype_map (source_title, doc_type_category, notes, added_at)
SELECT s.source_title, s.doc_type_category, 'Seeded from the former hardcoded view', current_timestamp()
FROM (VALUES
  ('Certified Copy of an Entry at Birth', 'BirthCertificate'),
  ('Certified Copy of an Entry of Marriage', 'MarriageCertificate'),
  ('Certified Copy of an Entry of Death', 'DeathCertificate'),
  ('Certified Copy of an entry at Death', 'DeathCertificate'),
  ('1841 England Census', 'Census'),
  ('1851 England Census', 'Census'),
  ('1861 England Census', 'Census'),
  ('1871 England Census', 'Census'),
  ('1881 England Census', 'Census'),
  ('1891 England Census', 'Census'),
  ('1901 England Census', 'Census'),
  ('1911 England Census', 'Census'),
  ('1921 England Census', 'Census'),
  ('1939 England and Wales Register', 'Census'),
  ('1841 Scotland Census', 'Census'),
  ('1851 Scotland Census', 'Census'),
  ('1861 Scotland Census', 'Census'),
  ('1871 Scotland Census', 'Census'),
  ('1881 Scotland Census', 'Census'),
  ('1891 Scotland Census', 'Census'),
  ('1901 Scotland Census', 'Census'),
  ('1911 Scotland Census', 'Census'),
  ('1921 Scotland Census', 'Census'),
  ('1850 United States Federal Census', 'Census'),
  ('1860 United States Federal Census', 'Census'),
  ('1870 United States Federal Census', 'Census'),
  ('1880 United States Federal Census', 'Census'),
  ('1890 United States Federal Census', 'Census'),
  ('1900 United States Federal Census', 'Census'),
  ('1910 United States Federal Census', 'Census'),
  ('1920 United States Federal Census', 'Census'),
  ('1930 United States Federal Census', 'Census'),
  ('1940 United States Federal Census', 'Census'),
  ('1950 United States Federal Census', 'Census'),
  ('1881 Census of Canada', 'Census'),
  ('1891 Census of Canada', 'Census'),
  ('1901 Census of Canada', 'Census'),
  ('1911 Census of Canada', 'Census'),
  ('1921 Census of Canada', 'Census'),
  ('1931 Census of Canada', 'Census'),
  ('England, Select Births and Christenings, 1538-1975', 'BirthIndex'),
  ('Scotland, Select Births and Baptisms, 1564-1950', 'BirthIndex'),
  ('England & Wales Christening Records, 1530-1906', 'BaptismRegister'),
  ('Nottinghamshire, England, Church of England Baptisms, 1813-1919', 'BaptismRegister'),
  ('Nottinghamshire, England, Church of England Baptisms, 1813-1922', 'BaptismRegister'),
  ('Cambridgeshire and Huntingdonshire, England, Church of England Births and Baptisms, 1813-1924', 'BaptismRegister'),
  ('Norfolk, England, Church of England Births and Baptisms, 1813-1919', 'BaptismRegister'),
  ('London, England, Births and Baptisms, 1813-1906', 'BaptismRegister'),
  ('Nottinghamshire, England, Church of England Marriages and Banns, 1754-1937', 'MarriageCertificate'),
  ('Scotland OPR Banns & Marriages, 1553-1854', 'MarriageCertificate'),
  ('Cambridgeshire and Huntingdonshire, England, Church of England Marriages and Banns, 1754-1950', 'MarriageCertificate'),
  ('Norfolk, England, Church of England Marriages and Banns, 1754-1936', 'MarriageCertificate'),
  ('Derbyshire, England, Church of England Marriages and Banns, 1754-1932', 'MarriageCertificate'),
  ('London, England, Marriages and Banns, 1754-1921', 'MarriageCertificate'),
  ('Sheffield and Rotherham, Yorkshire, England, Church of England Marriages and Banns, 1754-1948', 'MarriageCertificate'),
  ('Gretna Green, Scotland, Marriage Registers, 1794-1895', 'MarriageCertificate'),
  ('Nottinghamshire, England, Church of England Burials, 1813-1994', 'BurialRegister'),
  ('Cambridgeshire and Huntingdonshire, England, Church of England Deaths and Burials, 1813-1997', 'BurialRegister'),
  ('Norfolk, England, Church of England Deaths and Burials, 1813-1995', 'BurialRegister'),
  ('Web: UK, Burial and Cremation Index, 1576-2014', 'BurialRegister'),
  ('Scotland Statutory Death Register', 'DeathCertificate'),
  ('England & Wales, Birth Index: 1984-2005', 'BirthIndex'),
  ('England & Wales, FreeBMD Birth Index, 1837-1915', 'BirthIndex'),
  ('Scotland Statutory Register Births Index, 1855-', 'BirthIndex'),
  ('England & Wales, Marriage Index: 1916-2005', 'MarriageIndex'),
  ('England & Wales, Civil Registration Marriage Index, 1916-2005', 'MarriageIndex'),
  ('England & Wales, Civil Registration Marriage Index, 1837-1915', 'MarriageIndex'),
  ('England & Wales, FreeBMD Marriage Index: 1837-1915', 'MarriageIndex'),
  ('Scotland Statutory Register Marriages Index, 1855-1930', 'MarriageIndex'),
  ('England, Select Marriages, 1538-1973', 'MarriageIndex'),
  ('England, Select Marriages, 1538\u20131973', 'MarriageIndex'),
  ('Scotland, Select Marriages, 1561-1910', 'MarriageIndex'),
  ('England & Wales, Death Index: 1984-2005', 'DeathIndex'),
  ('England and Wales, Death Index, 1989-2019', 'DeathIndex'),
  ('England & Wales, Civil Registration Death Index, 1916-2007', 'DeathIndex'),
  ('England & Wales, FreeBMD Death Index: 1837-1915', 'DeathIndex'),
  ('England, Select Deaths and Burials, 1538-1991', 'DeathIndex'),
  ('Norfolk, England, Transcripts of Church of England Baptism, Marriage and Burial Registers, 1600-1935', 'OTHER'),
  ('British Phone Books, 1880-1984', 'OTHER_NO_FILE'),
  ('UK and Ireland, Find a Grave Index, 1300s-Current', 'OTHER_NO_FILE'),
  ('England & Wales, National Probate Calendar (Index of Wills and Administrations), 1858-1995', 'OTHER_NO_FILE'),
  ('Scotland, National Probate Index (Calendar of Confirmations and Inventories), 1876-1936', 'OTHER_NO_FILE'),
  ('U.K., City and County Directories, 1600s-1900s', 'OTHER_NO_FILE'),
  ('1911 England Census Summary Books', 'OTHER_NO_FILE'),
  ('West Yorkshire, England, Electoral Registers, 1840-1962', 'OTHER_NO_FILE'),
  ('Sheffield and Rotherham, Yorkshire, England, Church of England Baptisms, 1813-1923', 'BaptismRegister'),
  ('England & Wales, Civil Registration Birth Index, 1837-1915', 'BirthIndex'),
  ('UK, World War I Service Records, 1914-1920', 'MilitaryRecord'),
  ('UK, British Army World War I Service Records, 1914-1920', 'MilitaryRecord'),
  ('British Army WWI Service Records, 1914-1920', 'MilitaryRecord'),
  ('UK, World War II Service Records', 'MilitaryRecord'),
  ('UK, Soldiers Died in the Great War, 1914-1919', 'MilitaryRecord'),
  ('UK,  British Army World War I Pension Records 1914-1920', 'MilitaryRecord'),
  ('UK, Army Registers of Soldiers'' Effects, 1901-1929', 'MilitaryRecord'),
  ('Australia, World War II Military Service Records, 1939-1945', 'MilitaryRecord'),
  ('UK, Navy Lists, 1888-1970', 'MilitaryRecord'),
  ('UK, Royal Air Force Airmen Records, 1918-1940', 'MilitaryRecord'),
  ('UK and Ireland, Newspapers.com\u2122 Marriage Index, 1800s-current', 'NewspaperClipping'),
  ('U.S., Newspapers.com Obituary Index, 1800s-current', 'NewspaperClipping'),
  ('Canada, Newspapers.com Obituary Index, 1800s-current', 'NewspaperClipping'),
  ('UK and Ireland, Newspapers.com\u2122 Obituary Index, 1800s-current', 'NewspaperClipping'),
  ('UK, Electoral Registers, 2003-2010', 'OTHER_NO_FILE'),
  ('UK, Electoral Registers, 2011-2018', 'OTHER_NO_FILE'),
  ('Australia, Electoral Rolls, 1903-1980', 'ElectoralRegister'),
  ('New Zealand, Electoral Rolls, 1853-2010', 'ElectoralRegister'),
  ('London, England, Electoral Registers, 1832-1965', 'ElectoralRegister'),
  ('UK, Outward Passenger Lists, 1890-1960', 'PassengerList'),
  ('UK Incoming Passenger Lists, 1878-1960', 'PassengerList'),
  ('New York Passenger Lists, 1820-1957', 'PassengerList'),
  ('Pennsylvania, U.S., Arriving Passenger and Crew Lists, 1800-1962', 'PassengerList'),
  ('England & Wales, National Probate Calendar (Index of Wills and Administrations),1861-1941', 'Probate'),
  ('Scotland, Burial Monument Inscriptions, 1507-2019', 'MonumentalInscription'),
  ('North Lanarkshire, Scotland, Poor Law Applications and Registers, 1849-1917', 'PoorLawRecord'),
  ('UK, City and County Directories, 1766 - 1946', 'Directory'),
  ('England & Wales, Criminal Registers, 1791-1892', 'CriminalRecord'),
  ('Web: Scotland, Prison Records Index, 1828-1878', 'CriminalRecord'),
  ('UK, Calendar of Prisoners, 1868-1929', 'CriminalRecord')
) AS s(source_title, doc_type_category)
WHERE NOT EXISTS (SELECT 1 FROM genealogy.ref_source_doctype_map r WHERE r.source_title = s.source_title);

In [0]:
%sql
-- Asana 1218767063521380 changes (single-purpose titles only; combined collections and unclear titles stay unmapped).
INSERT INTO genealogy.ref_source_doctype_map (source_title, doc_type_category, notes, added_at)
SELECT s.source_title, s.doc_type_category, 'Added 2026-10-16, Asana 1218767063521380', current_timestamp()
FROM (VALUES
  ('Scotland Statutory Marriage Register, 1855-1934', 'MarriageCertificate'),
  ('Scotland Statutory Birth Register', 'BirthCertificate'),
  ('England & Wales, Civil Registration Death Index, 1837-1915', 'DeathIndex'),
  ('Pallot''s Marriage Index for England: 1780 - 1837', 'MarriageIndex'),
  ('England, Pallot''s Marriage Index, 1780-1837', 'MarriageIndex'),
  ('Scotland Statutory Register Deaths Index, 1855-1956', 'DeathIndex'),
  ('Norfolk, England, Church of England Marriages and Banns, 1754-1938', 'MarriageCertificate'),
  ('Norwich Death Returns (transcript)', 'DeathCertificate'),
  ('Transcription of Scotland Statutory Register Marriages Index, 1855-1930', 'MarriageIndex'),
  ('England & Wales Marriages, 1538-1940', 'MarriageIndex'),
  ('Certificate of Registry of Birth', 'BirthCertificate'),
  ('Derbyshire, England, Church of England Burials, 1813-1991', 'BurialRegister'),
  ('Norfolk Banns and Marriages', 'MarriageCertificate'),
  ('Derbyshire, England, Church of England Births and Baptisms, 1813-1916', 'BaptismRegister'),
  ('Scotland Statutory Marriages', 'MarriageCertificate'),
  ('Select Counties, Scotland, Church of Scotland, Marriages, 1615-1854', 'MarriageCertificate'),
  ('Scotland Old Parish Registers Births', 'BirthCertificate'),
  ('Derbyshire, England, Church of England Deaths and Burials, 1813-1991', 'BurialRegister'),
  ('Norfolk, England, Church of England Births and Baptisms, 1813-1922', 'BaptismRegister'),
  ('England & Wales, Christening Index, 1530-1980', 'BirthIndex'),
  ('Nottinghamshire Baptisms', 'BaptismRegister'),
  ('Pennsylvania, U.S., Death Certificates, 1906-1968', 'DeathCertificate'),
  ('UK, Lunacy Patients Admission Registers, 1846-1912', 'Other')
) AS s(source_title, doc_type_category)
WHERE NOT EXISTS (SELECT 1 FROM genealogy.ref_source_doctype_map r WHERE r.source_title = s.source_title);

In [0]:
%sql
-- Index, not the register entry: treat like the BMDIndex entries (INDEX_ONLY). If register images are bought, add a separate citation.
UPDATE genealogy.ref_source_doctype_map
SET doc_type_category = 'DeathIndex', notes = 'Changed from BurialRegister 2026-10-16, Asana 1218767063521380: index entry, not the register'
WHERE source_title = 'Web: UK, Burial and Cremation Index, 1576-2014' AND doc_type_category <> 'DeathIndex';

In [0]:
%sql
-- Check: expect 0 rows (duplicate or blank keys).
SELECT source_title, COUNT(*) AS n FROM genealogy.ref_source_doctype_map
GROUP BY source_title HAVING COUNT(*) > 1 OR TRIM(source_title) = '';